# Functional Form Comparison: Is Power Law the Right Model?

Tests whether coherence decay is best described by a power law vs alternative functional forms,
using AIC/BIC model selection on existing cached results.

**Candidate models**:
1. Power law: y = a * x^(-b)
2. Exponential: y = a * exp(-b*x)
3. Stretched exponential: y = a * exp(-b * x^c)
4. Logarithmic: y = a - b * log(x)
5. Linear: y = a - b*x

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, warnings, os
warnings.filterwarnings('ignore')

# === Mount Drive ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_BASE = Path('/content/drive/MyDrive/LRTIA/Results')
else:
    RESULTS_BASE = Path('../results')

print(f'Results base: {RESULTS_BASE}')
print(f'Exists: {RESULTS_BASE.exists()}')
if RESULTS_BASE.exists():
    print('Contents:', [d.name for d in sorted(RESULTS_BASE.iterdir()) if d.is_dir()])

In [ ]:
# === Discover all datasets ===
DATASETS = {}

# RAID (Mistral) human + AI
for pop in ['human', 'ai']:
    raid_ip = RESULTS_BASE / 'RAID_finegrain' / 'finegrain_results_v4.json'
    raid_sp = RESULTS_BASE / 'RAID_finegrain' / 'finegrain_shuffled_v4.json'
    if raid_ip.exists():
        DATASETS[f'RAID_{pop}_mistral'] = {
            'intact': raid_ip, 'shuffled': raid_sp,
            'filter_pop': pop, 'label': f'RAID {pop} (Mistral)',
        }

# Buckeye
bk_ip = RESULTS_BASE / 'Buckeye_finegrain' / 'buckeye_intact_v1.json'
bk_sp = RESULTS_BASE / 'Buckeye_finegrain' / 'buckeye_shuffled_v1.json'
if bk_ip.exists():
    DATASETS['Buckeye'] = {'intact': bk_ip, 'shuffled': bk_sp, 'label': 'Buckeye spoken'}

# French
fr_ip = RESULTS_BASE / 'French_oral_finegrain' / 'french_oral_intact_v1.json'
fr_sp = RESULTS_BASE / 'French_oral_finegrain' / 'french_oral_shuffled_v1.json'
if fr_ip.exists():
    DATASETS['French'] = {'intact': fr_ip, 'shuffled': fr_sp, 'label': 'French spoken'}

# Multilingual Wikipedia
for lang in ['zh', 'ja', 'ko', 'tr', 'ar', 'fi']:
    lip = RESULTS_BASE / 'Wiki_multilingual_finegrain' / f'{lang}_intact_v1.json'
    lsp = RESULTS_BASE / 'Wiki_multilingual_finegrain' / f'{lang}_shuffled_v1.json'
    if lip.exists():
        names = {'zh': 'Chinese', 'ja': 'Japanese', 'ko': 'Korean',
                 'tr': 'Turkish', 'ar': 'Arabic', 'fi': 'Finnish'}
        DATASETS[f'Wiki_{lang}'] = {'intact': lip, 'shuffled': lsp, 'label': f'{names[lang]} Wiki'}

# Multi-model
for mk in ['gpt2', 'gpt2-medium', 'llama3-8b']:
    for pop in ['human', 'ai']:
        mip = RESULTS_BASE / 'RAID_multimodel_finegrain' / f'{mk}_intact_v1.json'
        msp = RESULTS_BASE / 'RAID_multimodel_finegrain' / f'{mk}_shuffled_v1.json'
        if mip.exists():
            DATASETS[f'RAID_{pop}_{mk}'] = {
                'intact': mip, 'shuffled': msp,
                'filter_pop': pop, 'label': f'RAID {pop} ({mk})',
            }

print(f'Found {len(DATASETS)} datasets')
for k, v in DATASETS.items():
    print(f'  {v["label"]}')

In [ ]:
# === Load all corrected marginals ===
MAX_CONTEXT = 100
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def get_binned_marginals(corrected_marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = corrected_marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    return np.array(bc), np.array(bm)

all_marginals = {}

for key, info in DATASETS.items():
    with open(info['intact']) as f:
        intact = json.load(f)
    with open(info['shuffled']) as f:
        shuffled = json.load(f)
    if 'filter_pop' in info:
        pop = info['filter_pop']
        intact = [c for c in intact if c.get('population') == pop]
        shuffled = [c for c in shuffled if c.get('population') == pop]
    if len(intact) < 5 or len(shuffled) < 5:
        continue
    ip = compute_raw_ppl_curve(intact)
    sp = compute_raw_ppl_curve(shuffled)
    corr = -np.diff(ip) - (-np.diff(sp))
    bc, bm = get_binned_marginals(corr)
    if len(bc) >= 4:
        all_marginals[key] = {
            'bc': bc, 'bm': bm, 'raw_marg': corr,
            'label': info['label'], 'n': len(intact),
        }

print(f'Loaded {len(all_marginals)} datasets with valid corrected marginals')
for k, v in all_marginals.items():
    print(f'  {v["label"]} (n={v["n"]})')

In [ ]:
# === Define candidate functional forms + fitting ===

def power_law(x, a, b):
    return a * np.power(x, -b)

def exponential(x, a, b):
    return a * np.exp(-b * x)

def stretched_exp(x, a, b, c):
    return a * np.exp(-b * np.power(x, c))

def logarithmic(x, a, b):
    return a - b * np.log(x)

def linear(x, a, b):
    return a - b * x

MODELS = {
    'Power law':     {'func': power_law,     'p0': [1.0, 0.75], 'n_params': 2},
    'Exponential':   {'func': exponential,   'p0': [1.0, 0.05], 'n_params': 2},
    'Stretched exp': {'func': stretched_exp, 'p0': [1.0, 0.05, 0.5], 'n_params': 3},
    'Logarithmic':   {'func': logarithmic,   'p0': [1.0, 0.1],  'n_params': 2},
    'Linear':        {'func': linear,        'p0': [1.0, 0.01], 'n_params': 2},
}

def compute_aic(n, rss, k):
    if rss <= 0 or n <= k + 1:
        return np.inf
    return n * np.log(rss / n) + 2 * k

def compute_bic(n, rss, k):
    if rss <= 0 or n <= k + 1:
        return np.inf
    return n * np.log(rss / n) + k * np.log(n)

def fit_all_models(x, y):
    n = len(x)
    results = {}
    for name, spec in MODELS.items():
        try:
            bounds = (0, np.inf) if name != 'Logarithmic' else (-np.inf, np.inf)
            popt, pcov = optimize.curve_fit(
                spec['func'], x, y, p0=spec['p0'],
                maxfev=10000, bounds=bounds
            )
            y_pred = spec['func'](x, *popt)
            rss = np.sum((y - y_pred) ** 2)
            tss = np.sum((y - np.mean(y)) ** 2)
            r_squared = 1 - rss / tss if tss > 0 else 0
            k = spec['n_params']
            results[name] = {
                'params': popt, 'r_squared': r_squared, 'rss': rss,
                'aic': compute_aic(n, rss, k), 'bic': compute_bic(n, rss, k),
                'n_params': k, 'y_pred': y_pred, 'success': True,
            }
        except (RuntimeError, ValueError, TypeError):
            results[name] = {'success': False}
    successful = {k: v for k, v in results.items() if v.get('success')}
    if successful:
        best_aic = min(v['aic'] for v in successful.values())
        best_bic = min(v['bic'] for v in successful.values())
        for name in successful:
            results[name]['delta_aic'] = results[name]['aic'] - best_aic
            results[name]['delta_bic'] = results[name]['bic'] - best_bic
    return results

print('Fitting functions defined')

In [ ]:
# === Fit all models to all datasets ===
all_fits = {}
summary_rows = []

for key, data in all_marginals.items():
    fits = fit_all_models(data['bc'], data['bm'])
    all_fits[key] = fits
    successful = {k: v for k, v in fits.items() if v.get('success')}
    if not successful:
        continue
    best_aic_name = min(successful, key=lambda k: successful[k]['aic'])
    best_bic_name = min(successful, key=lambda k: successful[k]['bic'])
    for model_name, fit in fits.items():
        if not fit.get('success'):
            continue
        summary_rows.append({
            'dataset': key, 'label': data['label'], 'model': model_name,
            'r_squared': fit['r_squared'],
            'aic': fit['aic'], 'bic': fit['bic'],
            'delta_aic': fit.get('delta_aic', np.nan),
            'delta_bic': fit.get('delta_bic', np.nan),
            'n_params': fit['n_params'],
            'best_aic': model_name == best_aic_name,
            'best_bic': model_name == best_bic_name,
        })

df = pd.DataFrame(summary_rows)
print(f'Fit results: {len(df)} rows across {df["dataset"].nunique()} datasets')

if len(df) > 0:
    n_datasets = df['dataset'].nunique()
    aic_wins = df[df['best_aic']].groupby('model').size().sort_values(ascending=False)
    bic_wins = df[df['best_bic']].groupby('model').size().sort_values(ascending=False)
    print(f'\nAIC wins:')
    for model, count in aic_wins.items():
        print(f'  {model:<20}: {count:>3} / {n_datasets} ({count/n_datasets*100:.0f}%)')
    print(f'\nBIC wins:')
    for model, count in bic_wins.items():
        print(f'  {model:<20}: {count:>3} / {n_datasets} ({count/n_datasets*100:.0f}%)')

In [ ]:
# === Per-dataset breakdown ===
for key, data in all_marginals.items():
    fits = all_fits[key]
    successful = {k: v for k, v in fits.items() if v.get('success')}
    if not successful:
        continue
    print(f'\n--- {data["label"]} (n={data["n"]}) ---')
    print(f'  {"Model":<20} {"R²":>8} {"AIC":>10} {"ΔAIC":>8} {"BIC":>10} {"ΔBIC":>8}')
    print(f'  {"-"*72}')
    sorted_models = sorted(successful.items(), key=lambda x: x[1]['aic'])
    for name, fit in sorted_models:
        marker = ' ★' if fit.get('delta_aic', 99) == 0 else ''
        print(f'  {name:<20} {fit["r_squared"]:>8.4f} {fit["aic"]:>10.2f} '
              f'{fit.get("delta_aic", 0):>8.2f} {fit["bic"]:>10.2f} '
              f'{fit.get("delta_bic", 0):>8.2f}{marker}')

In [ ]:
# === Figure 1: Visual fits for up to 8 datasets ===
show_keys = list(all_marginals.keys())[:8]
n_show = len(show_keys)

if n_show == 0:
    print('No datasets to plot!')
else:
    ncols = min(4, n_show)
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes.reshape(1, -1)
    elif ncols == 1:
        axes = axes.reshape(-1, 1)
    axes_flat = axes.flatten()

    model_colors = {
        'Power law': '#1f77b4', 'Exponential': '#d62728',
        'Stretched exp': '#9467bd', 'Logarithmic': '#2ca02c', 'Linear': '#8c564b',
    }

    for idx, key in enumerate(show_keys):
        ax = axes_flat[idx]
        data = all_marginals[key]
        fits = all_fits[key]
        ax.plot(data['bc'], data['bm'], 'ko', markersize=8, zorder=10, label='Data')
        x_smooth = np.linspace(data['bc'].min(), data['bc'].max(), 200)
        successful = {k: v for k, v in fits.items() if v.get('success')}
        for name, fit in sorted(successful.items(), key=lambda x: x[1]['aic']):
            color = model_colors.get(name, 'gray')
            is_best = fit.get('delta_aic', 99) == 0
            style = '-' if is_best else '--'
            width = 2.5 if is_best else 1.0
            alpha = 1.0 if fit.get('delta_aic', 99) < 4 else 0.4
            try:
                y_smooth = MODELS[name]['func'](x_smooth, *fit['params'])
                lbl = f"{name} (R²={fit['r_squared']:.3f})"
                if is_best: lbl += ' ★'
                ax.plot(x_smooth, y_smooth, color=color, linestyle=style,
                        linewidth=width, alpha=alpha, label=lbl)
            except:
                pass
        ax.set_xscale('log')
        ax.set_xlabel('Distance (tokens)')
        ax.set_ylabel('Corrected Marginal')
        ax.set_title(data['label'], fontweight='bold', fontsize=11)
        ax.legend(fontsize=6, loc='upper right')
        ax.grid(True, alpha=0.2)
    for idx in range(n_show, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    plt.suptitle('Functional Form Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_BASE / 'functional_form_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# === Figure 2: Heatmap of ΔAIC ===
if len(df) == 0:
    print('No data for heatmap')
else:
    pivot_aic = df.pivot_table(index='label', columns='model', values='delta_aic')
    col_order = ['Power law', 'Exponential', 'Stretched exp', 'Logarithmic', 'Linear']
    pivot_aic = pivot_aic[[c for c in col_order if c in pivot_aic.columns]]

    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot_aic) * 0.4)))
    im = ax.imshow(pivot_aic.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=10)
    ax.set_xticks(range(len(pivot_aic.columns)))
    ax.set_xticklabels(pivot_aic.columns, fontsize=10, rotation=30, ha='right')
    ax.set_yticks(range(len(pivot_aic.index)))
    ax.set_yticklabels(pivot_aic.index, fontsize=9)
    for i in range(len(pivot_aic.index)):
        for j in range(len(pivot_aic.columns)):
            val = pivot_aic.values[i, j]
            if np.isnan(val): text = '—'
            elif val == 0: text = '★'
            else: text = f'{val:.1f}'
            color = 'white' if not np.isnan(val) and val > 5 else 'black'
            ax.text(j, i, text, ha='center', va='center', fontsize=8, color=color)
    plt.colorbar(im, ax=ax, label='ΔAIC (0=best)', shrink=0.8)
    ax.set_title('Model Selection: ΔAIC (★ = best)', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS_BASE / 'functional_form_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# === Final verdict ===
if len(df) == 0:
    print('No results to summarize')
else:
    n_datasets = df['dataset'].nunique()
    print('='*70)
    print('FINAL VERDICT')
    print('='*70)

    mean_r2 = df.groupby('model')['r_squared'].mean().sort_values(ascending=False)
    print('\nMean R² across all datasets:')
    for model, r2 in mean_r2.items():
        print(f'  {model:<20}: {r2:.4f}')

    mean_daic = df.groupby('model')['delta_aic'].mean().sort_values()
    print('\nMean ΔAIC (lower = better):')
    for model, daic in mean_daic.items():
        print(f'  {model:<20}: {daic:.2f}')

    aic_wins = df[df['best_aic']].groupby('model').size().sort_values(ascending=False)
    bic_wins = df[df['best_bic']].groupby('model').size().sort_values(ascending=False)
    print(f'\nAIC winner: {aic_wins.index[0]} ({aic_wins.iloc[0]}/{n_datasets} = {aic_wins.iloc[0]/n_datasets*100:.0f}%)')
    print(f'BIC winner: {bic_wins.index[0]} ({bic_wins.iloc[0]}/{n_datasets} = {bic_wins.iloc[0]/n_datasets*100:.0f}%)')